<a href="https://colab.research.google.com/github/Paras1719/GenAi-pracs/blob/main/GenAI_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Embedding, Dense
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [ ]:
english_sentences = [
    "hello",
    "how are you",
    "what is your name",
    "my name is john",
    "i am fine",
    "good morning",
    "good night",
    "thank you",
    "welcome",
    "where are you going",
    "i am going home",
    "i like india",
    "india is beautiful",
    "what are you doing",
    "i am learning",
    "i am a student",
    "this is my book",
    "this is my house",
    "open the door",
    "close the door",
    "please help me",
    "come here",
    "go there",
    "sit down",
    "stand up",
    "i love my family",
    "today is a good day",
    "what time is it",
    "i want water",
    "i want food",
    "where is the school",
    "where is the hospital",
    "i am happy",
    "i am sad",
    "do you understand",
    "yes i understand",
    "no i do not understand",
    "see you tomorrow",
    "see you soon",
    "have a nice day"
]

hindi_sentences = [
    "नमस्ते",
    "आप कैसे हैं",
    "आपका नाम क्या है",
    "मेरा नाम जॉन है",
    "मैं ठीक हूँ",
    "सुप्रभात",
    "शुभ रात्रि",
    "धन्यवाद",
    "स्वागत है",
    "आप कहाँ जा रहे हैं",
    "मैं घर जा रहा हूँ",
    "मुझे भारत पसंद है",
    "भारत सुंदर है",
    "आप क्या कर रहे हैं",
    "मैं सीख रहा हूँ",
    "मैं एक विद्यार्थी हूँ",
    "यह मेरी किताब है",
    "यह मेरा घर है",
    "दरवाजा खोलो",
    "दरवाजा बंद करो",
    "कृपया मेरी मदद करें",
    "यहाँ आओ",
    "वहाँ जाओ",
    "बैठ जाओ",
    "खड़े हो जाओ",
    "मैं अपने परिवार से प्यार करता हूँ",
    "आज एक अच्छा दिन है",
    "अभी कितने बजे हैं",
    "मुझे पानी चाहिए",
    "मुझे खाना चाहिए",
    "स्कूल कहाँ है",
    "अस्पताल कहाँ है",
    "मैं खुश हूँ",
    "मैं दुखी हूँ",
    "क्या आप समझते हैं",
    "हाँ मैं समझता हूँ",
    "नहीं मैं नहीं समझता",
    "कल मिलते हैं",
    "जल्द मिलते हैं",
    "आपका दिन शुभ हो"
]

In [ ]:
hindi_decoder_input = ["<start> " + s for s in hindi_sentences]
hindi_decoder_target = [s + " <end>" for s in hindi_sentences]

In [ ]:
english_tokenizer = Tokenizer()
english_tokenizer.fit_on_texts(english_sentences)

hindi_tokenizer = Tokenizer(filters="")
hindi_tokenizer.fit_on_texts(hindi_decoder_input + hindi_decoder_target)

english_sequences = english_tokenizer.texts_to_sequences(english_sentences)

decoder_input_sequences = hindi_tokenizer.texts_to_sequences(
    hindi_decoder_input
)

decoder_target_sequences = hindi_tokenizer.texts_to_sequences(
    hindi_decoder_target
)

In [ ]:
max_encoder_length = max(len(x) for x in english_sequences)
max_decoder_length = max(len(x) for x in decoder_input_sequences)

encoder_input_data = pad_sequences(
    english_sequences,
    maxlen=max_encoder_length,
    padding="post"
)

decoder_input_data = pad_sequences(
    decoder_input_sequences,
    maxlen=max_decoder_length,
    padding="post"
)

decoder_target_data = pad_sequences(
    decoder_target_sequences,
    maxlen=max_decoder_length,
    padding="post"
)

decoder_target_data = np.expand_dims(
    decoder_target_data,
    -1
)

In [ ]:
english_vocab_size = len(english_tokenizer.word_index) + 1
hindi_vocab_size = len(hindi_tokenizer.word_index) + 1

print("English Vocabulary Size:", english_vocab_size)
print("Hindi Vocabulary Size:", hindi_vocab_size)

print("Maximum English Length:", max_encoder_length)
print("Maximum Hindi Length:", max_decoder_length)

English Vocabulary Size: 70
Hindi Vocabulary Size: 76
Maximum English Length: 5
Maximum Hindi Length: 8


In [ ]:
embedding_dim = 128
latent_dim = 256

encoder_inputs = Input(
    shape=(None,),
    name="encoder_inputs"
)

encoder_embedding = Embedding(
    input_dim=english_vocab_size,
    output_dim=embedding_dim,
    name="encoder_embedding"
)(encoder_inputs)

encoder_lstm = LSTM(
    latent_dim,
    return_state=True,
    name="encoder_lstm"
)

encoder_outputs, state_h, state_c = encoder_lstm(
    encoder_embedding
)

encoder_states = [state_h, state_c]

In [ ]:
decoder_inputs = Input(
    shape=(None,),
    name="decoder_inputs"
)

decoder_embedding_layer = Embedding(
    input_dim=hindi_vocab_size,
    output_dim=embedding_dim,
    name="decoder_embedding"
)

decoder_embedding = decoder_embedding_layer(
    decoder_inputs
)

decoder_lstm = LSTM(
    latent_dim,
    return_sequences=True,
    return_state=True,
    name="decoder_lstm"
)

decoder_outputs, _, _ = decoder_lstm(
    decoder_embedding,
    initial_state=encoder_states
)

In [ ]:
decoder_dense = Dense(
    hindi_vocab_size,
    activation="softmax",
    name="decoder_output"
)

decoder_outputs = decoder_dense(
    decoder_outputs
)

In [ ]:
model = Model(
    [encoder_inputs, decoder_inputs],
    decoder_outputs
)

model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ encoder_inputs      │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_inputs      │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_embedding   │ (None, None, 128) │      8,960 │ encoder_inputs[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_embedding   │ (None, None, 128) │      9,728 │ decoder_inputs[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_lstm (LSTM) │ [(None, 256),     │    394,240 │ encoder_embeddin… │
│                     │ (None, 256),      │            │                   │
│                     │ (None, 256)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_lstm (LSTM) │ [(None, None,     │    394,240 │ decoder_embeddin… │
│                     │ 256), (None,      │            │ encoder_lstm[0][… │
│                     │ 256), (None,      │            │ encoder_lstm[0][… │
│                     │ 256)]             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_output      │ (None, None, 76)  │     19,532 │ decoder_lstm[0][… │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 826,700 (3.15 MB)

 Trainable params: 826,700 (3.15 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
history = model.fit(
    [encoder_input_data, decoder_input_data],
    decoder_target_data,
    batch_size=4,
    epochs=100,
    validation_split=0.2,
    verbose=1
)

Epoch 1/100
8/8 ━━━━━━━━━━━━━━━━━━━━ 6s 65ms/step - accuracy: 0.4062 - loss: 4.1531 - val_accuracy: 0.4375 - val_loss: 3.6258
Epoch 2/100
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.4648 - loss: 2.7308 - val_accuracy: 0.4375 - val_loss: 2.5730
Epoch 3/100
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.4648 - loss: 2.2610 - val_accuracy: 0.4375 - val_loss: 2.3452
Epoch 4/100
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.5508 - loss: 2.0275 - val_accuracy: 0.4688 - val_loss: 2.1347
Epoch 5/100
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.5078 - loss: 1.9257 - val_accuracy: 0.5625 - val_loss: 2.1072
Epoch 6/100
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.5898 - loss: 1.8178 - val_accuracy: 0.5625 - val_loss: 2.0811
Epoch 7/100
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.5898 - loss: 1.7721 - val_accuracy: 0.5625 - val_loss: 2.0658
Epoch 8/100
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.5898 - loss: 1.7290 - val_accuracy: 0.5625 - val_loss:

In [ ]:
encoder_model = Model(
    encoder_inputs,
    encoder_states
)

In [ ]:
decoder_state_input_h = Input(
    shape=(latent_dim,)
)

decoder_state_input_c = Input(
    shape=(latent_dim,)
)

decoder_states_inputs = [
    decoder_state_input_h,
    decoder_state_input_c
]

decoder_embedding_inf = decoder_embedding_layer(
    decoder_inputs
)

decoder_outputs_inf, state_h_inf, state_c_inf = decoder_lstm(
    decoder_embedding_inf,
    initial_state=decoder_states_inputs
)

decoder_states_inf = [
    state_h_inf,
    state_c_inf
]

decoder_outputs_inf = decoder_dense(
    decoder_outputs_inf
)

decoder_model = Model(
    [decoder_inputs] + decoder_states_inputs,
    [decoder_outputs_inf] + decoder_states_inf
)

In [ ]:
reverse_hindi_word_index = {
    value: key
    for key, value in hindi_tokenizer.word_index.items()
}

def translate_sentence(sentence):

    sequence = english_tokenizer.texts_to_sequences([sentence])

    sequence = pad_sequences(
        sequence,
        maxlen=max_encoder_length,
        padding="post"
    )

    states_value = encoder_model.predict(
        sequence,
        verbose=0
    )

    start_token = hindi_tokenizer.word_index["<start>"]

    end_token = hindi_tokenizer.word_index["<end>"]

    target_seq = np.array([[start_token]])

    decoded_sentence = []

    for _ in range(max_decoder_length):

        output_tokens, h, c = decoder_model.predict(
            [target_seq] + states_value,
            verbose=0
        )

        sampled_token_index = np.argmax(
            output_tokens[0, -1, :]
        )

        sampled_word = reverse_hindi_word_index.get(
            sampled_token_index,
            ""
        )

        if sampled_word == "<end>":
            break

        if sampled_word != "<start>" and sampled_word != "":
            decoded_sentence.append(sampled_word)

        target_seq = np.array([[sampled_token_index]])

        states_value = [h, c]

    return " ".join(decoded_sentence)

In [ ]:
test_sentences = [
    "hello",
    "how are you",
    "what is your name",
    "i am fine",
    "good morning",
    "thank you",
    "i am going home",
    "i like india",
    "what are you doing",
    "i am a student"
]

for sentence in test_sentences:

    translation = translate_sentence(sentence)

    print("English :", sentence)
    print("Hindi   :", translation)
    print("-" * 40)

English : hello
Hindi   : नमस्ते
----------------------------------------
English : how are you
Hindi   : आप कैसे हैं
----------------------------------------
English : what is your name
Hindi   : आपका नाम क्या है
----------------------------------------
English : i am fine
Hindi   : मैं ठीक हूँ
----------------------------------------
English : good morning
Hindi   : सुप्रभात
----------------------------------------
English : thank you
Hindi   : धन्यवाद
----------------------------------------
English : i am going home
Hindi   : मैं घर जा रहा हूँ
----------------------------------------
English : i like india
Hindi   : मुझे भारत पसंद है
----------------------------------------
English : what are you doing
Hindi   : आप क्या कर रहे हैं
----------------------------------------
English : i am a student
Hindi   : मैं एक विद्यार्थी हूँ
----------------------------------------


In [ ]:
while True:

    sentence = input("\nEnter English sentence (type 'exit' to stop): ")

    if sentence.lower() == "exit":
        break

    translation = translate_sentence(sentence)

    print("Hindi Translation:", translation)


Enter English sentence (type 'exit' to stop): english
Hindi Translation: नमस्ते

Enter English sentence (type 'exit' to stop): water
Hindi Translation: यहाँ आओ

Enter English sentence (type 'exit' to stop): pen
Hindi Translation: नमस्ते


KeyboardInterrupt: Interrupted by user